# Transformer fine-tuning for Hinglish sentiment (Colab)

**Not run in the build sandbox** (no GPU, no Hugging Face access), so run it yourself on Colab: *Runtime -> Change runtime type -> T4 GPU*.

Goal: compare fine-tuned transformers against the TF-IDF baselines from `src/semeval.py` on SemEval-2020 Task 9 (Hinglish). Reference scores from the task paper: mBERT baseline 65.4, best system 75.0 weighted F1.

Setup used by the task baseline: AdamW, lr 2e-5, 3 epochs, max length about 56 tokens.

Models to compare: `xlm-roberta-base`, `l3cube-pune/hing-roberta` (XLM-R further pre-trained on romanized Hinglish), `bert-base-multilingual-cased`.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

In [ ]:
# Upload the SemEval Hinglish train/dev files (from https://zenodo.org/record/3974927) via the Colab file panel, then set the paths:
TRAIN = 'train.txt'   # <- adjust to your file names
DEV   = 'dev.txt'

import pandas as pd
def parse_conll(path):
    rows, toks, lab = [], [], None
    for line in open(path, encoding='utf-8', errors='ignore').read().splitlines():
        if line.startswith('meta'):
            lab = line.split('\t')[-1].strip().lower(); toks = []
        elif not line.strip():
            if lab and toks: rows.append((' '.join(toks), lab))
            toks, lab = [], None
        else:
            toks.append(line.split('\t')[0])
    if lab and toks: rows.append((' '.join(toks), lab))
    return pd.DataFrame(rows, columns=['text', 'label'])

tr, dv = parse_conll(TRAIN), parse_conll(DEV)
LABELS = ['negative', 'neutral', 'positive']
l2i = {l: i for i, l in enumerate(LABELS)}
print(len(tr), len(dv), tr.label.value_counts().to_dict())   # check the files parsed correctly!

In [ ]:
import numpy as np, torch
from datasets import Dataset
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer)

def run(model_name, epochs=3, lr=2e-5, max_len=56, bs=32, seed=42):
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = lambda d: Dataset.from_dict({'text': d.text.tolist(), 'label': d.label.map(l2i).tolist()}).map(
        lambda b: tok(b['text'], truncation=True, max_length=max_len, padding='max_length'), batched=True)
    dtr, ddv = enc(tr), enc(dv)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
    def metrics(p):
        pred = p.predictions.argmax(-1)
        return {'weighted_f1': f1_score(p.label_ids, pred, average='weighted'), 'macro_f1': f1_score(p.label_ids, pred, average='macro')}
    args = TrainingArguments(output_dir=f'out_{model_name.replace("/", "_")}', num_train_epochs=epochs, learning_rate=lr,
        per_device_train_batch_size=bs, per_device_eval_batch_size=64, eval_strategy='epoch', save_strategy='no',
        weight_decay=0.01, seed=seed, fp16=torch.cuda.is_available(), report_to='none')
    t = Trainer(model=model, args=args, train_dataset=dtr, eval_dataset=ddv, compute_metrics=metrics)
    t.train()
    return t.evaluate()

results = {}
for name in ['bert-base-multilingual-cased', 'xlm-roberta-base', 'l3cube-pune/hing-roberta']:
    results[name] = run(name)
    print(name, results[name])

In [ ]:
pd.DataFrame(results).T[['eval_weighted_f1', 'eval_macro_f1']].round(4)
# Put this table next to the TF-IDF results from `python src/semeval.py` and the paper's reference scores.

## What to report
- Table: TF-IDF + NB/SVM/KNN vs mBERT vs XLM-R vs HingRoBERTa (weighted F1 on dev).
- Run each transformer with 3 seeds and report mean +/- std (one run can mislead).
- Short error analysis: which class is confused most (neutral usually), and example mistakes.
- Do not tune on the official test set.